# 🤖 Mô hình Baseline: TF-IDF + Logistic Regression

Dự án môn Trí Tuệ Nhân Tạo – Nhóm Giday  
**Thành viên thực hiện:** Hiền (TV1)  
**Nhiệm vụ:** Xây dựng mô hình Baseline sử dụng phương pháp TF-IDF để trích chọn đặc trưng và thuật toán Logistic Regression để phân loại nhằm đối chiếu và so sánh với mô hình học sâu PhoBERT.

In [ ]:
# Cài đặt các thư viện cần thiết nếu chạy trên Colab
!pip install scikit-learn pandas numpy matplotlib seaborn huggingface_hub py_vncorenlp underthesea -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import unicodedata
import os
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from huggingface_hub import login

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'

## 📂 1. Tải và Tiền xử lý dữ liệu

Mô hình Baseline sẽ sử dụng dữ liệu đã tách từ (segmented) từ bước tiền xử lý để đảm bảo tính nhất quán.
Chúng ta sẽ ưu tiên tải file checkpoint `data_segmented.csv` đã qua tiền xử lý bằng VnCoreNLP.
Nếu chạy trên môi trường Colab mới mà chưa có file checkpoint, notebook sẽ tự động tải dữ liệu gốc từ HuggingFace và thực hiện tiền xử lý (clean text + tách từ).

In [ ]:
# 1. Định nghĩa các hàm tiền xử lý (giống hệt Tiền_xử_lý.ipynb)
def clean_text(text: object) -> str:
    if text is None or pd.isna(text):
        return ""
    text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\S+|www\S+", " <URL> ", text)
    text = re.sub(r"\S+@\S+", " <EMAIL> ", text)
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Cố gắng load data_segmented.csv trước
csv_filename = "data_segmented.csv"
if os.path.exists(csv_filename):
    print(f"✅ Tìm thấy file '{csv_filename}'. Tiến hành load dữ liệu...")
    df = pd.read_csv(csv_filename)
else:
    print(f"⚠️ Không tìm thấy '{csv_filename}'. Tiến hành tải dữ liệu từ HuggingFace...")
    # Tải dataset trực tiếp
    # Bạn có thể login bằng token Hugging Face của mình nếu cần:
    # login(token="YOUR_HF_TOKEN")
    splits = {
        'train': 'data/train-00000-of-00001.parquet',
        'dev': 'data/dev-00000-of-00001.parquet',
        'test': 'data/test-00000-of-00001.parquet'
    }
    try:
        df_train = pd.read_parquet("hf://datasets/tranthaihoa/vifactcheck/" + splits["train"])
        df_dev = pd.read_parquet("hf://datasets/tranthaihoa/vifactcheck/" + splits["dev"])
        df_test = pd.read_parquet("hf://datasets/tranthaihoa/vifactcheck/" + splits["test"])
        
        df_train['split'] = 'train'
        df_dev['split'] = 'dev'
        df_test['split'] = 'test'
        
        df = pd.concat([df_train, df_dev, df_test], ignore_index=True)
        print(f"Tổng số mẫu tải về: {len(df)}")
        
        # Tiền xử lý văn bản sạch
        print("Đang làm sạch văn bản...")
        df['Statement'] = df['Statement'].apply(clean_text)
        df['Evidence'] = df['Evidence'].apply(clean_text)
        
        # Thử tách từ
        print("Đang tiến hành tách từ (word segmentation)...")
        try:
            import py_vncorenlp
            vncorenlp_dir = './vncorenlp'
            save_dir = os.path.abspath(vncorenlp_dir)
            if not os.path.exists(save_dir) or len(os.listdir(save_dir)) == 0:
                os.makedirs(save_dir, exist_ok=True)
                py_vncorenlp.download_model(save_dir=save_dir)
            rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=save_dir)
            
            def word_segment(text):
                if not text or len(text.strip()) == 0:
                    return text
                sentences = rdrsegmenter.word_segment(text)
                return ' '.join(sentences)
            
            df['Statement_seg'] = df['Statement'].apply(word_segment)
            df['Evidence_seg'] = df['Evidence'].apply(word_segment)
            print("✅ Tách từ thành công bằng VnCoreNLP!")
        except Exception as ve:
            print(f"⚠️ Lỗi VnCoreNLP: {ve}. Chuyển sang dùng underthesea...")
            try:
                from underthesea import word_tokenize
                def segment_uts(text):
                    if not text:
                        return ""
                    return " ".join([w.replace(" ", "_") for w in word_tokenize(text)])
                df['Statement_seg'] = df['Statement'].apply(segment_uts)
                df['Evidence_seg'] = df['Evidence'].apply(segment_uts)
                print("✅ Tách từ thành công bằng underthesea!")
            except Exception as ue:
                print(f"⚠️ Lỗi underthesea: {ue}. Sử dụng phân tách khoảng trắng mặc định.")
                df['Statement_seg'] = df['Statement']
                df['Evidence_seg'] = df['Evidence']
                
        # Lưu checkpoint để tái sử dụng
        df.to_csv("data_segmented.csv", index=False)
        print("💾 Đã lưu checkpoint 'data_segmented.csv'")
    except Exception as ex:
        print(f"❌ Không thể tải hoặc xử lý dữ liệu: {ex}")

## 📊 2. Phân chia dữ liệu Train / Dev / Test

Chúng ta phân tách lại dữ liệu dựa vào cột `split` đã gán.

In [ ]:
# Tách các tập dữ liệu
train_df = df[df['split'] == 'train'].reset_index(drop=True)
dev_df = df[df['split'] == 'dev'].reset_index(drop=True)
test_df = df[df['split'] == 'test'].reset_index(drop=True)

print(f"Số mẫu Train: {len(train_df)}")
print(f"Số mẫu Dev  : {len(dev_df)}")
print(f"Số mẫu Test : {len(test_df)}")

# Kiểm tra phân bố nhãn trong tập train
print("\nPhân bố nhãn trong tập Train (0: SUPPORTED, 1: REFUTED, 2: NEI):")
print(train_df['labels'].value_counts(normalize=True).round(4) * 100)

## ✍️ 3. Trích chọn đặc trưng bằng TF-IDF

Mô hình sẽ kết hợp cả **Statement** (Tuyên bố) và **Evidence** (Bằng chứng) để đưa ra dự đoán.
Cách tiếp cận chính: Ghép cặp Statement và Evidence lại với nhau cách nhau bởi khoảng trắng, sau đó đưa qua `TfidfVectorizer`.

In [ ]:
# Tạo text ghép cặp cho cả 3 tập
train_df['combined_text'] = train_df['Statement_seg'].astype(str) + " " + train_df['Evidence_seg'].astype(str)
dev_df['combined_text'] = dev_df['Statement_seg'].astype(str) + " " + dev_df['Evidence_seg'].astype(str)
test_df['combined_text'] = test_df['Statement_seg'].astype(str) + " " + test_df['Evidence_seg'].astype(str)

# Khởi tạo TF-IDF Vectorizer
# Sử dụng uni-gram và bi-gram, lọc các từ xuất hiện quá ít (min_df=2)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000)

# Fit trên tập train và transform cả 3 tập
X_train = vectorizer.fit_transform(train_df['combined_text'])
X_dev = vectorizer.transform(dev_df['combined_text'])
X_test = vectorizer.transform(test_df['combined_text'])

y_train = train_df['labels'].values
y_dev = dev_df['labels'].values
y_test = test_df['labels'].values

print(f"Kích thước ma trận đặc trưng Train: {X_train.shape}")
print(f"Kích thước ma trận đặc trưng Dev  : {X_dev.shape}")
print(f"Kích thước ma trận đặc trưng Test : {X_test.shape}")

## 🧠 4. Huấn luyện mô hình Logistic Regression

Chúng ta sử dụng Logistic Regression làm baseline phân loại 3 nhãn. 
Vì dữ liệu có sự mất cân bằng giữa các nhãn, chúng ta sử dụng tham số `class_weight='balanced'` để tự động điều chỉnh trọng số các nhãn.

In [ ]:
# Khởi tạo mô hình Logistic Regression
# Tăng max_iter để đảm bảo hội tụ và đặt class_weight='balanced'
model = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=42)

# Huấn luyện mô hình
print("Đang huấn luyện mô hình...")
model.fit(X_train, y_train)
print("✅ Huấn luyện hoàn tất!")

## 📈 5. Đánh giá hiệu năng mô hình

Đánh giá mô hình trên cả tập **Validation (Dev)** và **Test** với các độ đo Accuracy, Precision, Recall, và F1-score.

In [ ]:
# Hàm đánh giá và in báo cáo chi tiết
def evaluate_model(X, y_true, split_name="Dev"):
    y_pred = model.predict(X)
    
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro')
    f1_weighted = f1_score(y_true, y_pred, average='weighted')
    
    print(f"=== ĐÁNH GIÁ TRÊN TẬP {split_name.upper()} ===")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Macro: {f1_macro:.4f}")
    print(f"F1 Weighted: {f1_weighted:.4f}\n")
    
    target_names = ['SUPPORTED (0)', 'REFUTED (1)', 'NEI (2)']
    print(classification_report(y_true, y_pred, target_names=target_names))
    return y_pred, acc, f1_macro

print("--- Kết quả trên tập Validation (Dev) ---")
y_pred_dev, acc_dev, f1_dev = evaluate_model(X_dev, y_dev, "Dev")

print("\n--- Kết quả trên tập Test ---")
y_pred_test, acc_test, f1_test = evaluate_model(X_test, y_test, "Test")

## 🗺️ 6. Confusion Matrix (Ma trận nhầm lẫn)

Vẽ biểu đồ Confusion Matrix để phân tích xem mô hình thường bị nhầm lẫn ở những nhãn nào.

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred)
    labels = ['SUPPORTED', 'REFUTED', 'NEI']
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('Nhãn dự đoán (Predicted)', fontsize=12)
    plt.ylabel('Nhãn thực tế (Actual)', fontsize=12)
    plt.tight_layout()
    plt.show()

print("Biểu đồ Confusion Matrix trên tập Dev:")
plot_confusion_matrix(y_dev, y_pred_dev, "Confusion Matrix - Dev Set")

print("Biểu đồ Confusion Matrix trên tập Test:")
plot_confusion_matrix(y_test, y_pred_test, "Confusion Matrix - Test Set")

## 🔬 7. Ablation Study (Nghiên cứu loại trừ)

Để hiểu tầm quan trọng của từng thành phần văn bản (Claim/Statement và Evidence) đối với bài toán kiểm tra tính xác thực, chúng ta thực hiện Ablation Study trên 3 trường hợp:
1. **Chỉ sử dụng Statement (Claim-only)**
2. **Chỉ sử dụng Evidence (Evidence-only)**
3. **Sử dụng kết hợp cả hai (Combined)**

In [ ]:
results_ablation = []

for case_name, train_col, dev_col, test_col in [
    ("Claim-only (Statement)", "Statement_seg", "Statement_seg", "Statement_seg"),
    ("Evidence-only", "Evidence_seg", "Evidence_seg", "Evidence_seg"),
    ("Combined (Claim + Evidence)", "combined_text", "combined_text", "combined_text")
]:
    print(f"\nEvaluating case: {case_name}...")
    
    # Vectorize
    vectorizer_ab = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000)
    X_tr = vectorizer_ab.fit_transform(train_df[train_col].astype(str))
    X_de = vectorizer_ab.transform(dev_df[dev_col].astype(str))
    X_te = vectorizer_ab.transform(test_df[test_col].astype(str))
    
    # Train
    model_ab = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=42)
    model_ab.fit(X_tr, y_train)
    
    # Dev Predict
    preds_de = model_ab.predict(X_de)
    acc_de = accuracy_score(y_dev, preds_de)
    f1_de_macro = f1_score(y_dev, preds_de, average='macro')
    
    # Test Predict
    preds_te = model_ab.predict(X_te)
    acc_te = accuracy_score(y_test, preds_te)
    f1_te_macro = f1_score(y_test, preds_te, average='macro')
    
    results_ablation.append({
        "Thử nghiệm": case_name,
        "Dev Acc": acc_de,
        "Dev F1 Macro": f1_de_macro,
        "Test Acc": acc_te,
        "Test F1 Macro": f1_te_macro
    })

# Tạo DataFrame hiển thị kết quả
df_results = pd.DataFrame(results_ablation)
display(df_results.round(4))